In [1]:
# @title 🚀 1. 환경 설정 및 라이브러리 설치
# 필수 라이브러리 설치 (evaluate 포함)
!pip install -q evaluate transformers accelerate scikit-learn librosa soundfile

import os
import shutil
import random
import numpy as np
import librosa
import torch
import evaluate
from google.colab import drive
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import classification_report, confusion_matrix

# 드라이브 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("✅ 환경 설정 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
Mounted at /content/drive
✅ 환경 설정 완료!


In [2]:
# @title 📦 2. 데이터셋 셋업 (Python 기반 안정화 버전)
import os
import shutil
import glob

# ==========================================
# 🔧 경로 설정
# ==========================================
DRIVE_ZIP_PATH = "/content/drive/MyDrive/HabitLink/Data.zip"
LOCAL_ZIP_PATH = "/content/Data.zip"
EXTRACT_PATH = "/content/Data" # 압축을 풀 폴더

# ==========================================
# 🚀 실행 로직
# ==========================================
print("🔄 1. 데이터셋 복사 중... (Drive -> Colab Local)")

# 드라이브 파일 확인 및 예비 경로 체크
if not os.path.exists(DRIVE_ZIP_PATH):
    ALT_PATH = "/content/drive/MyDrive/HabitLink/dataset.zip"
    if os.path.exists(ALT_PATH):
        print(f"⚠️ 'Data.zip'이 없어 '{ALT_PATH}'를 사용합니다.")
        DRIVE_ZIP_PATH = ALT_PATH
    else:
        raise FileNotFoundError(f"❌ 드라이브에 zip 파일이 없습니다!\n경로: {DRIVE_ZIP_PATH}")

# 파일 복사 (이미 있으면 건너뜀)
if not os.path.exists(LOCAL_ZIP_PATH):
    shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    print("✅ 복사 완료!")
else:
    print("ℹ️ 이미 복사된 파일이 있어 건너뜁니다.")

print("\n📦 2. 압축 해제 및 정리 중... (안전 모드)")

# 기존 데이터 폴더가 있다면 깨끗하게 삭제 후 재생성
if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)
os.makedirs(EXTRACT_PATH, exist_ok=True)

# 압축 해제 (os.system 사용 - 아까 잘 되던 방식)
# -q: 조용히, -d: 풀 경로 지정
exit_code = os.system(f"unzip -q {LOCAL_ZIP_PATH} -d {EXTRACT_PATH}")

if exit_code != 0:
    raise RuntimeError("❌ 압축 해제 중 오류가 발생했습니다.")

# (중요) 맥북 쓰레기 파일 제거
os.system(f'rm -rf {EXTRACT_PATH}/__MACOSX')
os.system(f'find {EXTRACT_PATH} -name "._*" -delete')

print("✅ 데이터셋 준비 완료!")

# ==========================================
# 📊 3. 데이터 검증 (잘 풀렸는지 확인)
# ==========================================
print("\n📊 [데이터 검증]")
search_root = EXTRACT_PATH

# 압축 푼 구조 확인 (이중 폴더 방지)
sub_dirs = [d for d in os.listdir(EXTRACT_PATH) if os.path.isdir(os.path.join(EXTRACT_PATH, d))]
if len(sub_dirs) == 1 and "Data" in sub_dirs: # Data 폴더 안에 Data가 또 있는 경우
    search_root = os.path.join(EXTRACT_PATH, "Data")
    print(f"ℹ️ 루트 폴더 보정됨: {search_root}")

target_folders = ["standard", "gyeongsang", "jeolla_jeju", "chungcheong_gangwon"]
total_count = 0

for folder in target_folders:
    folder_path = os.path.join(search_root, folder)
    if os.path.exists(folder_path):
        count = len(glob.glob(os.path.join(folder_path, "*.wav")))
        print(f"  - {folder:<20}: {count}개")
        total_count += count
    else:
        print(f"  ⚠️ {folder:<20}: 폴더 없음")

print("-" * 30)
print(f"🎉 총 데이터 개수: {total_count}개")

🔄 1. 데이터셋 복사 중... (Drive -> Colab Local)
✅ 복사 완료!

📦 2. 압축 해제 및 정리 중... (안전 모드)
✅ 데이터셋 준비 완료!

📊 [데이터 검증]
ℹ️ 루트 폴더 보정됨: /content/Data/Data
  - standard            : 12000개
  - gyeongsang          : 4000개
  - jeolla_jeju         : 4000개
  - chungcheong_gangwon : 4000개
------------------------------
🎉 총 데이터 개수: 24000개


In [3]:
# @title 🔍 3. 경로 자동 탐색 및 설정 (최적화 버전)
import os
import torch

def find_data_root(start_dir="/content/Data"): # 👈 탐색 범위를 확 줄였습니다!
    print(f"🔍 '{start_dir}' 내부에서 'standard' 폴더를 찾는 중...")

    if not os.path.exists(start_dir):
        print(f"⚠️ 경고: '{start_dir}' 폴더가 없습니다. '/content'에서 다시 찾습니다.")
        start_dir = "/content" # 혹시 없으면 전체 탐색 (예비책)

    for root, dirs, files in os.walk(start_dir):
        if "standard" in dirs:
            print(f"✅ 데이터 발견! 루트 경로: {root}")
            return root

    return None

# 경로 자동 설정
ROOT_DIR = find_data_root()

if ROOT_DIR is None:
    # 혹시 못 찾았을 경우 디버깅 정보 출력
    print("❌ 'standard' 폴더를 찾지 못했습니다.")
    print("📂 현재 /content/Data 폴더 구조:")
    if os.path.exists("/content/Data"):
        print(os.listdir("/content/Data"))
    else:
        print("'/content/Data' 폴더 자체가 없습니다.")
    raise FileNotFoundError("압축은 풀렸으나 데이터 폴더 구조가 예상과 다릅니다.")

# 결과 모델 저장 경로 (구글 드라이브)
OUTPUT_DIR = "/content/drive/MyDrive/HabitLink/models/dialect_binary_classifier_final"

# 하이퍼파라미터 (A100 GPU 최적화)
BASE_MODEL = "kresnik/wav2vec2-large-xlsr-korean"
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 3e-5
MAX_DURATION = 3.0

# GPU 확인
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ 사용 디바이스: {device}")
print(f"📂 학습 데이터 경로: {ROOT_DIR}")
print(f"💾 모델 저장 경로: {OUTPUT_DIR}")

🔍 '/content/Data' 내부에서 'standard' 폴더를 찾는 중...
✅ 데이터 발견! 루트 경로: /content/Data/Data
🖥️ 사용 디바이스: cuda
📂 학습 데이터 경로: /content/Data/Data
💾 모델 저장 경로: /content/drive/MyDrive/HabitLink/models/dialect_binary_classifier_final


In [4]:
# @title 📂 4. 데이터 로드 및 분할
def load_files(folder_name):
    path = os.path.join(ROOT_DIR, folder_name)
    if not os.path.exists(path):
        print(f"⚠️ 경고: '{folder_name}' 폴더가 없습니다.")
        return []

    files = []
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.lower().endswith(('.wav', '.mp3', '.flac', '.m4a')):
                files.append(entry.path)
    return files

print("🔄 파일 리스트 읽는 중...")

# 각 지역별 파일 로드
std_files = load_files("standard")
gs_files = load_files("gyeongsang")
jj_files = load_files("jeolla_jeju")
cg_files = load_files("chungcheong_gangwon")

# 데이터셋 통합 (Label 0: 표준어, Label 1: 방언)
all_data = []
for f in std_files:
    all_data.append({"path": f, "label": 0})

for f in gs_files + jj_files + cg_files:
    all_data.append({"path": f, "label": 1})

# 데이터 개수 체크
if len(all_data) == 0:
    raise ValueError("❌ 로드된 데이터가 0개입니다! 경로를 확인하세요.")

# 섞기
random.seed(42)
random.shuffle(all_data)

print(f"\n📊 데이터 구성 완료")
print(f"  - 표준어 (Label 0): {len(std_files)}개")
print(f"  - 방언   (Label 1): {len(gs_files) + len(jj_files) + len(cg_files)}개")
print(f"  👉 총 데이터 개수 : {len(all_data)}개")

# 분할 (80% / 10% / 10%)
train_data, temp_data = train_test_split(all_data, test_size=0.2, stratify=[x['label'] for x in all_data], random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=[x['label'] for x in temp_data], random_state=42)

print(f"\n✂️ 데이터 분할 완료")
print(f"  - Train Set : {len(train_data)}개")
print(f"  - Valid Set : {len(val_data)}개")
print(f"  - Test Set  : {len(test_data)}개")

🔄 파일 리스트 읽는 중...

📊 데이터 구성 완료
  - 표준어 (Label 0): 12000개
  - 방언   (Label 1): 12000개
  👉 총 데이터 개수 : 24000개

✂️ 데이터 분할 완료
  - Train Set : 19200개
  - Valid Set : 2400개
  - Test Set  : 2400개


In [5]:
# @title 🔧 5. 데이터셋 클래스 (Augmentation 적용)
class DialectDataset(Dataset):
    def __init__(self, data_list, processor, max_duration=3.0, is_train=False):
        self.data_list = data_list
        self.processor = processor
        self.max_duration = max_duration
        self.sr = 16000
        self.is_train = is_train

    def __len__(self):
        return len(self.data_list)

    def add_white_noise(self, audio):
        # 랜덤 노이즈 추가
        noise_factor = np.random.uniform(0.005, 0.02)
        noise = np.random.randn(len(audio))
        augmented_audio = audio + noise_factor * noise
        return augmented_audio.astype(type(audio[0]))

    def __getitem__(self, idx):
        item = self.data_list[idx]
        file_path = item['path']
        label = item['label']

        try:
            speech, _ = librosa.load(file_path, sr=self.sr, duration=self.max_duration)
            if len(speech) < 100: # 너무 짧은 파일 처리
                speech = np.zeros(int(self.sr * self.max_duration))

            # 학습 모드일 때만 노이즈 추가
            if self.is_train and random.random() < 0.8:
                speech = self.add_white_noise(speech)

            inputs = self.processor(
                speech, sampling_rate=self.sr, max_length=int(self.sr * self.max_duration),
                truncation=True, padding="max_length", return_tensors="pt"
            )

            return {
                "input_values": inputs.input_values[0],
                "labels": torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            return {"input_values": torch.zeros(int(self.sr * self.max_duration)), "labels": torch.tensor(label, dtype=torch.long)}

In [6]:
# @title 🚀 6. 모델 학습 실행
# 모델 & 프로세서 로드
print("🤖 모델 로드 중...")
processor = Wav2Vec2Processor.from_pretrained(BASE_MODEL)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0: "Standard", 1: "Non-Standard"},
    label2id={"Standard": 0, "Non-Standard": 1}
)
model.to(device)

# 데이터셋 연결
train_dataset = DialectDataset(train_data, processor, MAX_DURATION, is_train=True)
val_dataset = DialectDataset(val_data, processor, MAX_DURATION, is_train=False)
test_dataset = DialectDataset(test_data, processor, MAX_DURATION, is_train=False)

# 평가 메트릭
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Trainer 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    save_total_limit=2, fp16=True, dataloader_num_workers=2
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🔥 학습 시작! (Validation 포함)")
trainer.train()

print("\n💾 최적 모델 저장 중...")
trainer.save_model(f"{OUTPUT_DIR}/best_model")
processor.save_pretrained(f"{OUTPUT_DIR}/best_model")
print(f"✅ 저장 완료: {OUTPUT_DIR}/best_model")

🤖 모델 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Accuracy
1,0.638200,0.599900,0.672083
2,0.592600,0.556693,0.716250
3,0.547800,0.531436,0.748333
4,0.512100,0.511541,0.769583
5,0.467600,0.497842,0.774167



💾 최적 모델 저장 중...
✅ 저장 완료: /content/drive/MyDrive/HabitLink/models/dialect_binary_classifier_final/best_model


In [7]:
# @title 📊 7. 최종 테스트 평가
print("\n📊 테스트 데이터셋 평가 중...")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("\n=== Classification Report ===")
print(classification_report(labels, preds, target_names=["Standard", "Non-Standard"]))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(labels, preds))


📊 테스트 데이터셋 평가 중...



=== Classification Report ===
              precision    recall  f1-score   support

    Standard       0.75      0.79      0.77      1200
Non-Standard       0.78      0.74      0.76      1200

    accuracy                           0.76      2400
   macro avg       0.77      0.76      0.76      2400
weighted avg       0.77      0.76      0.76      2400


=== Confusion Matrix ===
[[951 249]
 [317 883]]


In [12]:
# @title 🎤 새로운 오디오 파일 디버깅 & 추론 (Inference Debug Mode)
import os
import librosa
import torch
import numpy as np
import IPython.display as ipd
import torch.nn.functional as F

# 1. 테스트할 파일 경로 설정
# (Colab 파일 탐색기에서 경로를 복사해 넣으세요)
TEST_FILE_PATH = "/content/drive/MyDrive/HabitLink/test/표준어.m4a"

def predict_dialect_with_debug(file_path):
    print(f"📄 분석 파일: {os.path.basename(file_path)}")

    # 1. 오디오 로드
    # sr=16000은 Wav2Vec 2.0 모델의 필수 조건입니다.
    try:
        speech, sr = librosa.load(file_path, sr=16000, duration=3.0) # 3초만 로드
    except Exception as e:
        print(f"❌ 오디오 로드 실패: {e}")
        return

    # 2. [디버깅] 모델에게 입력되는 소리 들어보기
    # 만약 여기서 소리가 '치지직'거리거나, 속도가 너무 빠르거나 느리면 전처리 문제입니다.
    print("\n⬇️ [Check] 모델이 듣게 될 소리 (재생해보세요) ⬇️")
    ipd.display(ipd.Audio(speech, rate=16000))

    # 3. 전처리
    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    # 4. GPU로 이동 및 추론
    model.eval()
    with torch.no_grad():
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(**inputs).logits

    # 5. [디버깅] 결과 해석 (Softmax로 확률 계산)
    probs = F.softmax(logits, dim=-1)[0]

    # id2label을 이용해 라벨 매핑 (0: Standard, 1: Non-Standard)
    # 모델 config에 저장된 라벨 순서를 따릅니다.
    label_0_name = model.config.id2label[0] # 보통 Standard
    label_1_name = model.config.id2label[1] # 보통 Non-Standard

    score_0 = probs[0].item() * 100
    score_1 = probs[1].item() * 100

    predicted_class_id = torch.argmax(logits, dim=-1).item()
    predicted_label = model.config.id2label[predicted_class_id]

    print(f"\n📊 [분석 결과]")
    print(f"   👉 최종 예측: [{predicted_label}]")
    print(f"   -------------------------------")
    print(f"   1️⃣ {label_0_name:<12}: {score_0:.2f}%")
    print(f"   2️⃣ {label_1_name:<12}: {score_1:.2f}%")

    # 6. [진단] 과적합(Overfitting) 여부 판단
    print(f"\n🩺 [AI 진단]")
    if predicted_label == "Non-Standard" and "표준어" in file_path:
        if score_1 > 99.0:
            print("🚨 [심각] 모델이 표준어를 '방언'이라고 99% 이상 확신하고 있습니다.")
            print("   -> 학습 데이터의 '특정 잡음(Noise)'이나 '마이크 특성'을 외워버렸을 가능성이 매우 높습니다.")
            print("   -> 해결책: 데이터 증강(Noise 추가) 후 재학습이 필요합니다.")
        elif score_1 > 60.0:
            print("🤔 [주의] 모델이 애매하게 방언으로 판단했습니다.")
            print("   -> 말투, 억양, 혹은 목소리 톤이 학습된 방언 데이터와 비슷할 수 있습니다.")
        else:
            print("❓ [양호] 틀렸지만 확신도가 높지 않습니다. 학습 데이터 부족일 수 있습니다.")
    elif predicted_label == "Standard":
        print("✅ [정상] 표준어를 정확하게 표준어로 분류했습니다.")
    else:
         print("ℹ️ 결과 확인 완료.")

# 실행
if os.path.exists(TEST_FILE_PATH):
    predict_dialect_with_debug(TEST_FILE_PATH)
else:
    print(f"⚠️ 파일이 존재하지 않습니다: {TEST_FILE_PATH}")
    print("경로를 다시 확인해주세요.")

📄 분석 파일: 표준어.m4a

⬇️ [Check] 모델이 듣게 될 소리 (재생해보세요) ⬇️


/tmp/ipython-input-323964463.py:19: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, sr = librosa.load(file_path, sr=16000, duration=3.0) # 3초만 로드
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



📊 [분석 결과]
   👉 최종 예측: [Non-Standard]
   -------------------------------
   1️⃣ Standard    : 7.81%
   2️⃣ Non-Standard: 92.19%

🩺 [AI 진단]
🤔 [주의] 모델이 애매하게 방언으로 판단했습니다.
   -> 말투, 억양, 혹은 목소리 톤이 학습된 방언 데이터와 비슷할 수 있습니다.
